<a href="https://colab.research.google.com/github/Arnab-apk/Agentic_Development/blob/arnab/first_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
%%capture
# 1. Force uninstall everything to clear conflicts
!pip uninstall -y langchain langchain-community langchain-core langchain-openai openai duckduckgo-search

# 2. Install the specific compatible versions
!pip install langchain==0.1.16 langchain-community==0.0.34 langchain-core==0.1.45 langchain-openai==0.1.3 openai duckduckgo-search

In [6]:
import os
from langchain_openai import ChatOpenAI
from langchain_community.agent_toolkits.load_tools import load_tools
from langchain_core.tools import Tool
from langchain_community.tools import DuckDuckGoSearchResults
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate

# 1. SETUP API KEY
os.environ["OPENAI_API_KEY"] = "sk-..." # <-- PASTE YOUR KEY HERE

# 2. SETUP LLM
openai_llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

# 3. SETUP TOOLS
search = DuckDuckGoSearchResults()
search_tool = Tool(
    name="duckduck",
    description="A web search engine. Use this to search for current events.",
    func=search.run,
)
tools = load_tools(["llm-math"], llm=openai_llm)
tools.append(search_tool)

# 4. SETUP PROMPT
react_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate(
    template=react_template,
    input_variables=["tools", "tool_names", "input", "agent_scratchpad"]
)

# 5. RUN AGENT
agent = create_react_agent(openai_llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

# Test
agent_executor.invoke({"input": "What is 15% of 500?"})

ImportError: cannot import name 'AgentExecutor' from 'langchain.agents' (/usr/local/lib/python3.12/dist-packages/langchain/agents/__init__.py)